In [1]:
!pip install langchain langchain-groq duckduckgo-search chromadb pydantic -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

In [2]:
import os
import json
import time
import logging
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from typing import Optional
from pydantic import BaseModel, field_validator
from google.colab import userdata

# ── Logging ───────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# ── Config ────────────────────────────────────────────────────
TICKER = "AAPL"
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

# ── Groq client ───────────────────────────────────────────────
from groq import Groq
client = Groq(api_key=GROQ_API_KEY)

# ── JSONL Trace Logger ────────────────────────────────────────
# This handles Task 3C observability requirement
TRACE_FILE = "agent_trace.jsonl"

def log_tool_call(tool_name: str, inputs: dict, output: any, duration: float):
    """Logs every tool call to agent_trace.jsonl for observability."""
    entry = {
        "timestamp": datetime.now().isoformat(),
        "tool_name": tool_name,
        "inputs": inputs,
        "output": str(output)[:200],   # truncated to 200 chars as required
        "duration_seconds": round(duration, 3)
    }
    with open(TRACE_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")
    logger.info(f"🔧 Tool called: {tool_name} ({duration:.2f}s)")

print("✅ Foundation setup complete")

✅ Foundation setup complete


In [7]:
# ============================================================
# THE 5 AGENT TOOLS — Task 3A
# ============================================================

# ── Tool 1: get_price_data ────────────────────────────────────
def get_price_data(ticker: str, period: str = "2y") -> dict:
    """
    Wraps yfinance. Returns OHLCV data with computed indicators.
    """
    start = time.time()
    try:
        end_date = datetime.today()
        start_date = end_date - timedelta(days=365 * 2)
        df = yf.download(ticker, start=start_date, end=end_date,
                         auto_adjust=True, progress=False)

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()

        # Compute indicators
        df["SMA_50"]  = df["Close"].rolling(50).mean()
        df["SMA_200"] = df["Close"].rolling(200).mean()

        delta = df["Close"].diff()
        gain  = delta.clip(lower=0)
        loss  = -delta.clip(upper=0)
        avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
        avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
        df["RSI_14"] = 100 - (100 / (1 + avg_gain / avg_loss))

        ema_12 = df["Close"].ewm(span=12, adjust=False).mean()
        ema_26 = df["Close"].ewm(span=26, adjust=False).mean()
        df["MACD"]        = ema_12 - ema_26
        df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()

        roll = df["Close"].rolling(20)
        df["BB_Upper"] = roll.mean() + 2 * roll.std()
        df["BB_Lower"] = roll.mean() - 2 * roll.std()

        latest = df.iloc[-1]
        result = {
            "ticker": ticker,
            "current_price": round(float(latest["Close"]), 2),
            "SMA_50":        round(float(latest["SMA_50"]), 2),
            "SMA_200":       round(float(latest["SMA_200"]), 2),
            "RSI_14":        round(float(latest["RSI_14"]), 2),
            "MACD":          round(float(latest["MACD"]), 4),
            "MACD_Signal":   round(float(latest["MACD_Signal"]), 4),
            "BB_Upper":      round(float(latest["BB_Upper"]), 2),
            "BB_Lower":      round(float(latest["BB_Lower"]), 2),
            "rows_fetched":  len(df)
        }
        log_tool_call("get_price_data", {"ticker": ticker, "period": period},
                      result, time.time() - start)
        return result

    except Exception as e:
        logger.error(f"get_price_data failed: {e}")
        log_tool_call("get_price_data", {"ticker": ticker}, f"ERROR: {e}",
                      time.time() - start)
        return {"error": str(e)}


# ── Tool 2: get_news ──────────────────────────────────────────
def get_news(ticker: str, n: int = 10) -> list:
    """
    Retrieves recent headlines for a ticker using yfinance.
    """
    start = time.time()
    try:
        raw = yf.Ticker(ticker).news or []
        headlines = []
        for item in raw[:n]:
            content = item.get("content", {})
            title   = content.get("title", "").strip()
            if title:
                headlines.append({
                    "title":     title,
                    "source":    content.get("provider", {}).get("displayName", "Unknown"),
                    "published": content.get("pubDate", "")
                })

        log_tool_call("get_news", {"ticker": ticker, "n": n},
                      f"{len(headlines)} headlines", time.time() - start)
        return headlines

    except Exception as e:
        logger.error(f"get_news failed: {e}")
        log_tool_call("get_news", {"ticker": ticker}, f"ERROR: {e}",
                      time.time() - start)
        return []


# ── Tool 3: calculate_volatility ─────────────────────────────
def calculate_volatility(ticker: str, window: int = 30) -> dict:
    """
    Computes annualised historical volatility from daily log returns.
    """
    start = time.time()
    try:
        end_date   = datetime.today()
        start_date = end_date - timedelta(days=window * 3)
        df = yf.download(ticker, start=start_date, end=end_date,
                         auto_adjust=True, progress=False)

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # Log returns → annualised volatility (252 trading days)
        log_returns   = np.log(df["Close"] / df["Close"].shift(1)).dropna()
        daily_vol     = float(log_returns.tail(window).std())
        annual_vol    = round(daily_vol * np.sqrt(252) * 100, 2)
        daily_vol_pct = round(daily_vol * 100, 4)

        result = {
            "ticker":             ticker,
            "window_days":        window,
            "daily_volatility":   daily_vol_pct,
            "annual_volatility":  annual_vol,
            "interpretation":     (
                "High volatility" if annual_vol > 40
                else "Moderate volatility" if annual_vol > 20
                else "Low volatility"
            )
        }
        log_tool_call("calculate_volatility",
                      {"ticker": ticker, "window": window},
                      result, time.time() - start)
        return result

    except Exception as e:
        logger.error(f"calculate_volatility failed: {e}")
        log_tool_call("calculate_volatility", {"ticker": ticker},
                      f"ERROR: {e}", time.time() - start)
        return {"error": str(e)}


# ── Tool 4: llm_sentiment ─────────────────────────────────────
def llm_sentiment(headlines: list) -> dict:
    """
    Calls Groq LLM and returns a structured sentiment score.
    """
    start = time.time()
    try:
        if not headlines:
            return {"error": "No headlines provided"}

        headline_text = "\n".join(
            f"- {h['title'] if isinstance(h, dict) else h}"
            for h in headlines
        )

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": (
                    "You are a financial sentiment analyst. "
                    "Analyse the provided headlines and return ONLY a JSON object with: "
                    "overall_sentiment (positive/negative/neutral), "
                    "confidence (0.0-1.0), "
                    "score (-1.0 to 1.0), "
                    "summary (one sentence). "
                    "No markdown, no explanation."
                )},
                {"role": "user", "content": headline_text}
            ],
            temperature=0.1,
            max_tokens=200,
        )

        raw    = response.choices[0].message.content.strip()
        result = json.loads(raw)

        log_tool_call("llm_sentiment",
                      {"headline_count": len(headlines)},
                      result, time.time() - start)
        return result

    except Exception as e:
        logger.error(f"llm_sentiment failed: {e}")
        log_tool_call("llm_sentiment", {"headline_count": len(headlines)},
                      f"ERROR: {e}", time.time() - start)
        return {"error": str(e), "overall_sentiment": "neutral", "score": 0.0}


# ── Tool 5: web_search ────────────────────────────────────────
def web_search(query: str) -> list:
    """
    Uses ddgs to retrieve analyst commentary.
    """
    start = time.time()
    try:
        from ddgs import DDGS
        results = []
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=5):
                results.append({
                    "title":   r.get("title", ""),
                    "snippet": r.get("body", "")[:300],
                    "url":     r.get("href", "")
                })

        log_tool_call("web_search", {"query": query},
                      f"{len(results)} results", time.time() - start)
        return results

    except Exception as e:
        logger.error(f"web_search failed: {e}")
        log_tool_call("web_search", {"query": query},
                      f"ERROR: {e}", time.time() - start)
        return []

# Test it
search_results = web_search(f"{TICKER} stock analyst outlook 2026")
print(f"Results: {len(search_results)}")
if search_results:
    print(f"Sample: {search_results[0]['title'][:70]}")


print("✅ All 5 tools defined")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Results: 5
Sample: Apple (AAPL) Stock Forecast and Price Target 2026 $AAPL
✅ All 5 tools defined


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [5]:
# ============================================================
# QUICK TEST — All 5 Tools
# ============================================================

print("=" * 55)
print("TESTING ALL 5 TOOLS")
print("=" * 55)

# Tool 1
print("\n[1] get_price_data...")
price_data = get_price_data(TICKER)
print(f"    Price: ${price_data['current_price']} | RSI: {price_data['RSI_14']} | MACD: {price_data['MACD']}")

# Tool 2
print("\n[2] get_news...")
news = get_news(TICKER, n=5)
print(f"    Headlines fetched: {len(news)}")
print(f"    Sample: {news[0]['title'][:60] if news else 'None'}")

# Tool 3
print("\n[3] calculate_volatility...")
vol = calculate_volatility(TICKER, window=30)
print(f"    Annual Vol: {vol['annual_volatility']}% — {vol['interpretation']}")

# Tool 4
print("\n[4] llm_sentiment...")
sentiment = llm_sentiment(news)
print(f"    Sentiment: {sentiment.get('overall_sentiment')} | Score: {sentiment.get('score')}")
print(f"    Summary: {sentiment.get('summary', '')[:80]}")

# Tool 5
print("\n[5] web_search...")
search_results = web_search(f"{TICKER} stock analyst outlook 2026")
print(f"    Results fetched: {len(search_results)}")
print(f"    Sample: {search_results[0]['title'][:60] if search_results else 'None'}")

print("\n" + "=" * 55)
print("✅ All tools working — ready to build the agent!")
print("=" * 55)

TESTING ALL 5 TOOLS

[1] get_price_data...
    Price: $280.14 | RSI: 66.44 | MACD: 4.3613

[2] get_news...
    Headlines fetched: 5
    Sample: Dow Jones Futures: Trump Says U.S. To 'Guide' Ships Through 

[3] calculate_volatility...
    Annual Vol: 23.81% — Moderate volatility

[4] llm_sentiment...
    Sentiment: positive | Score: 0.5
    Summary: The overall market sentiment is positive due to record-breaking Asian stocks and

[5] web_search...


/tmp/ipykernel_10528/288099292.py:197: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


    Results fetched: 0
    Sample: None

✅ All tools working — ready to build the agent!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [6]:
!pip install ddgs -q

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 2.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [8]:
# ============================================================
# TASK 3A — Single Research Agent
# ============================================================

# Short-term memory — stores results within a session
session_memory = {}

def run_research_agent(ticker: str) -> dict:
    """
    Autonomous research agent that decides which tools to call
    and in what order based on observations.
    """
    print(f"\n{'='*60}")
    print(f"🤖 RESEARCH AGENT STARTING — {ticker}")
    print(f"{'='*60}\n")

    # ── Step 1: Get price data first (always needed) ──────────
    print("📊 [OBSERVE] Fetching price data...")
    price_data = get_price_data(ticker)
    session_memory["price_data"] = price_data
    print(f"   Price: ${price_data['current_price']} | RSI: {price_data['RSI_14']}")

    # ── Step 2: Agent decides next action based on RSI ────────
    # If RSI > 65, agent flags potential overbought — needs more context
    if price_data["RSI_14"] > 65:
        print("\n🔍 [DECIDE] RSI elevated — fetching volatility to assess risk...")
        vol_data = calculate_volatility(ticker, window=30)
        session_memory["volatility"] = vol_data
        print(f"   Annual Vol: {vol_data['annual_volatility']}% — {vol_data['interpretation']}")
    else:
        print("\n🔍 [DECIDE] RSI normal — skipping deep volatility check...")
        vol_data = calculate_volatility(ticker, window=30)
        session_memory["volatility"] = vol_data

    # ── Step 3: Get news ──────────────────────────────────────
    print("\n📰 [OBSERVE] Fetching news headlines...")
    news = get_news(ticker, n=10)
    session_memory["news"] = news
    print(f"   {len(news)} headlines retrieved")

    # ── Step 4: Agent observes news count — decides sentiment ─
    if len(news) >= 3:
        print("\n💬 [DECIDE] Sufficient headlines — running sentiment analysis...")
        sentiment = llm_sentiment(news)
        session_memory["sentiment"] = sentiment
        print(f"   Sentiment: {sentiment.get('overall_sentiment')} | Score: {sentiment.get('score')}")
    else:
        print("\n⚠️ [DECIDE] Too few headlines — skipping sentiment, using neutral...")
        sentiment = {"overall_sentiment": "neutral", "score": 0.0, "summary": "Insufficient data"}
        session_memory["sentiment"] = sentiment

    # ── Step 5: Web search for analyst commentary ─────────────
    print(f"\n🌐 [OBSERVE] Searching for analyst commentary on {ticker}...")
    search_results = web_search(f"{ticker} stock analyst outlook risks 2026")
    session_memory["web_search"] = search_results

    # ── Step 6: If search fails, try alternative query ────────
    if not search_results:
        print("   ⚠️ First search empty — trying alternative query...")
        search_results = web_search(f"{ticker} stock forecast")
        session_memory["web_search"] = search_results

    print(f"   {len(search_results)} results found")

    # ── Step 7: Generate final report via LLM ─────────────────
    print("\n📝 [SYNTHESISE] Generating final research report...")

    context = f"""
Ticker: {ticker}
Current Price: ${price_data['current_price']}
RSI (14): {price_data['RSI_14']}
MACD: {price_data['MACD']} (Signal: {price_data['MACD_Signal']})
SMA 50: {price_data['SMA_50']} | SMA 200: {price_data['SMA_200']}
BB Upper: {price_data['BB_Upper']} | BB Lower: {price_data['BB_Lower']}
Annual Volatility: {vol_data['annual_volatility']}% ({vol_data['interpretation']})
News Sentiment: {sentiment.get('overall_sentiment')} (score: {sentiment.get('score')})
Sentiment Summary: {sentiment.get('summary', 'N/A')}
Recent Headlines: {'; '.join([n['title'] for n in news[:3]])}
Analyst Commentary: {'; '.join([r['snippet'][:100] for r in search_results[:2]])}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": (
                "You are a senior equity research analyst. "
                "Based on the data provided, produce a structured report with exactly "
                "three sections:\n"
                "1. FINANCIAL HEALTH SUMMARY — current technical position\n"
                "2. TOP THREE RISKS — each with supporting evidence from the data\n"
                "3. HEDGE STRATEGY RECOMMENDATION — one data-driven hedge\n"
                "Be specific and reference the actual numbers provided."
            )},
            {"role": "user", "content": context}
        ],
        temperature=0.3,
        max_tokens=800,
    )

    report = response.choices[0].message.content.strip()
    session_memory["final_report"] = report

    print("\n" + "="*60)
    print("📋 FINAL RESEARCH REPORT")
    print("="*60)
    print(report)
    print("="*60)

    return {
        "ticker": ticker,
        "price_data": price_data,
        "volatility": vol_data,
        "sentiment": sentiment,
        "report": report
    }


# ── Run the agent ─────────────────────────────────────────────
agent_result = run_research_agent(TICKER)
print(f"\n✅ Agent completed — {len(session_memory)} items stored in session memory")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


🤖 RESEARCH AGENT STARTING — AAPL

📊 [OBSERVE] Fetching price data...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

   Price: $280.14 | RSI: 66.44

🔍 [DECIDE] RSI elevated — fetching volatility to assess risk...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

   5 results found

📝 [SYNTHESISE] Generating final research report...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


📋 FINAL RESEARCH REPORT
## FINANCIAL HEALTH SUMMARY
AAPL's current technical position indicates a bullish trend. The Relative Strength Index (RSI) of 66.44 suggests that the stock is approaching overbought territory, but has not yet reached extreme levels. The Moving Average Convergence Divergence (MACD) of 4.3613, with a signal line of 3.4548, indicates a positive momentum. The stock price of $280.14 is above both the 50-day Simple Moving Average (SMA) of 261.22 and the 200-day SMA of 254.9, further supporting the bullish trend. Additionally, the stock is currently trading near the upper Bollinger Band (BB Upper: 280.06), which may indicate a potential pullback.

## TOP THREE RISKS
1. **Overbought Conditions**: With an RSI of 66.44, AAPL is approaching overbought territory, which may lead to a correction or pullback in the stock price. If the RSI reaches extreme levels (above 70), it could be a sign that the stock is due for a reversal.
2. **Volatility**: The annual volatility of 23.

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [9]:
# ============================================================
# TASK 3B — Two-Agent Pipeline + TASK 3C — Memory & Cache
# ============================================================

# ── Pydantic schema for structured agent handoff ──────────────
class DataBrief(BaseModel):
    """Structured handoff from Agent A to Agent B."""
    ticker: str
    current_price: float
    annual_volatility: float
    volatility_interpretation: str
    rsi: float
    macd: float
    macd_signal: float
    sma_50: float
    sma_200: float
    sentiment_score: float
    sentiment_label: str
    sentiment_summary: str
    momentum: str  # BULLISH / BEARISH / NEUTRAL
    clarification_response: Optional[str] = None

# ── Persistent cache helpers (Task 3C) ────────────────────────
CACHE_DIR = "research_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def get_cache_path(ticker: str) -> str:
    date_str = datetime.now().strftime("%Y-%m-%d")
    return f"{CACHE_DIR}/{ticker}_{date_str}.json"

def load_cache(ticker: str) -> Optional[dict]:
    path = get_cache_path(ticker)
    if os.path.exists(path):
        with open(path, "r") as f:
            print(f"📂 [CACHE] Found existing brief for {ticker} — loading...")
            return json.load(f)
    return None

def save_cache(ticker: str, data: dict):
    path = get_cache_path(ticker)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"💾 [CACHE] Research brief saved to {path}")

# ============================================================
# AGENT A — Data Analyst (quantitative tools only)
# ============================================================
def run_agent_a(ticker: str) -> DataBrief:
    print(f"\n{'='*60}")
    print(f"🤖 AGENT A (Data Analyst) — {ticker}")
    print(f"{'='*60}")

    # Tool access: get_price_data, calculate_volatility, llm_sentiment
    print("📊 Fetching price data...")
    price = get_price_data(ticker)

    print("📈 Calculating volatility...")
    vol = calculate_volatility(ticker, window=30)

    print("💬 Running sentiment analysis...")
    news = get_news(ticker, n=10)
    sentiment = llm_sentiment(news) if news else {"overall_sentiment": "neutral", "score": 0.0, "summary": "No data"}

    # Determine momentum
    bullish = price["SMA_50"] > price["SMA_200"] and price["MACD"] > price["MACD_Signal"]
    momentum = "BULLISH" if bullish else "BEARISH" if price["MACD"] < price["MACD_Signal"] else "NEUTRAL"

    brief = DataBrief(
        ticker=ticker,
        current_price=price["current_price"],
        annual_volatility=vol["annual_volatility"],
        volatility_interpretation=vol["interpretation"],
        rsi=price["RSI_14"],
        macd=price["MACD"],
        macd_signal=price["MACD_Signal"],
        sma_50=price["SMA_50"],
        sma_200=price["SMA_200"],
        sentiment_score=float(sentiment.get("score", 0.0)),
        sentiment_label=sentiment.get("overall_sentiment", "neutral"),
        sentiment_summary=sentiment.get("summary", ""),
        momentum=momentum
    )

    print(f"\n✅ Agent A complete — handing off structured brief to Agent B")
    print(f"   Price: ${brief.current_price} | Vol: {brief.annual_volatility}% | Momentum: {brief.momentum}")
    return brief


# ============================================================
# AGENT B — Research Writer (web_search, get_news only)
# ============================================================
def run_agent_b(brief: DataBrief) -> dict:
    print(f"\n{'='*60}")
    print(f"🤖 AGENT B (Research Writer) — {brief.ticker}")
    print(f"{'='*60}")

    # Tool access: web_search, get_news only — no price tools
    print("🌐 Searching for qualitative context...")
    search1 = web_search(f"{brief.ticker} stock risks analyst 2026")
    search2 = web_search(f"{brief.ticker} company news recent developments")

    print("📰 Fetching latest headlines...")
    news = get_news(brief.ticker, n=5)

    # ── Critique loop: Agent B requests clarification from Agent A ──
    print("\n🔄 [CRITIQUE LOOP] Agent B requesting clarification from Agent A...")
    clarification_request = f"Can you confirm: is the current RSI of {brief.rsi} trending up or down over the past 5 days? And is volatility of {brief.annual_volatility}% above or below the 90-day average?"

    print(f"   Agent B asks: '{clarification_request[:80]}...'")

    # Agent A responds (simulated with LLM using data it has)
    clarification_response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are Agent A, a quantitative analyst. Answer the clarification request concisely using the data provided."},
            {"role": "user", "content": f"Data: RSI={brief.rsi}, MACD={brief.macd}, Signal={brief.macd_signal}, Vol={brief.annual_volatility}%. Question: {clarification_request}"}
        ],
        temperature=0.1,
        max_tokens=150,
    ).choices[0].message.content.strip()

    brief.clarification_response = clarification_response
    print(f"   Agent A responds: '{clarification_response[:100]}...'")
    print("   Agent B incorporating clarification into final report...")

    # ── Agent B generates final report ────────────────────────
    print("\n📝 Agent B generating final research report...")

    analyst_snippets = " | ".join([r["snippet"][:100] for r in search1[:2]] + [r["snippet"][:100] for r in search2[:2]])
    headline_titles  = " | ".join([n["title"] for n in news[:3]])

    context = f"""
QUANTITATIVE BRIEF (from Agent A):
- Ticker: {brief.ticker} | Price: ${brief.current_price}
- Volatility: {brief.annual_volatility}% ({brief.volatility_interpretation})
- RSI: {brief.rsi} | MACD: {brief.macd} vs Signal: {brief.macd_signal}
- SMA50: {brief.sma_50} | SMA200: {brief.sma_200}
- Sentiment: {brief.sentiment_label} (score: {brief.sentiment_score})
- Momentum: {brief.momentum}
- Clarification from Agent A: {brief.clarification_response}

QUALITATIVE CONTEXT (from web search):
{analyst_snippets}

RECENT HEADLINES:
{headline_titles}
"""
    report = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": (
                "You are Agent B, a senior research writer. Using the quantitative brief from Agent A "
                "and the qualitative web research, write a professional research report with three sections:\n"
                "1. FINANCIAL HEALTH SUMMARY\n"
                "2. TOP THREE RISKS (with evidence from both quantitative and qualitative sources)\n"
                "3. HEDGE STRATEGY RECOMMENDATION\n"
                "Reference specific numbers from Agent A's brief and specific findings from web research."
            )},
            {"role": "user", "content": context}
        ],
        temperature=0.3,
        max_tokens=900,
    ).choices[0].message.content.strip()

    print("\n" + "="*60)
    print("📋 FINAL MULTI-AGENT RESEARCH REPORT")
    print("="*60)
    print(report)
    print("="*60)

    return {
        "ticker": brief.ticker,
        "agent_a_brief": brief.model_dump(),
        "final_report": report,
        "generated_at": datetime.now().isoformat()
    }


# ============================================================
# TASK 3C — Short-term memory demo + persistent cache
# ============================================================
def run_followup_question(question: str) -> str:
    """Answers follow-up using session memory — no re-fetching tools."""
    print(f"\n🧠 [SHORT-TERM MEMORY] Follow-up: '{question}'")
    print("   Using cached session memory — no tool calls made")

    context = json.dumps({k: str(v)[:200] for k, v in session_memory.items()})

    answer = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Answer the question using only the provided session memory data. Do not make up information."},
            {"role": "user", "content": f"Session memory: {context}\n\nQuestion: {question}"}
        ],
        temperature=0.1,
        max_tokens=200,
    ).choices[0].message.content.strip()

    print(f"   Answer: {answer}")
    return answer


# ============================================================
# RUN EVERYTHING
# ============================================================

# Check persistent cache first (Task 3C)
cached = load_cache(TICKER)
if cached:
    print("✅ Loaded from cache — skipping agent pipeline")
    final_result = cached
else:
    # Run two-agent pipeline (Task 3B)
    brief = run_agent_a(TICKER)
    final_result = run_agent_b(brief)

    # Save to persistent cache (Task 3C)
    save_cache(TICKER, final_result)

# Short-term memory demo (Task 3C)
run_followup_question("What was the RSI value and what does it suggest about the stock?")

# Save agent_trace.jsonl to verify it exists
print(f"\n📁 agent_trace.jsonl entries: {sum(1 for _ in open(TRACE_FILE))}")
print("✅ Task 3B and 3C complete!")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


🤖 AGENT A (Data Analyst) — AAPL
📊 Fetching price data...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

📈 Calculating volatility...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

   Agent A responds: 'To confirm the trend of the RSI and assess volatility, I would need historical RSI data and the 90-d...'
   Agent B incorporating clarification into final report...

📝 Agent B generating final research report...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


📋 FINAL MULTI-AGENT RESEARCH REPORT
**Research Report: Apple Inc. (AAPL)**

### 1. FINANCIAL HEALTH SUMMARY
Apple Inc. (AAPL) is currently trading at $280.14, with a moderate volatility of 23.81%. The Relative Strength Index (RSI) stands at 66.44, indicating a potential uptrend, although historical data is required to confirm the trend's direction. The Moving Average Convergence Divergence (MACD) is 4.3613, surpassing the signal line of 3.4548, which supports the bullish momentum. The 50-day Simple Moving Average (SMA50) is $261.22, and the 200-day Simple Moving Average (SMA200) is $254.9, both of which are below the current price, further reinforcing the positive sentiment. The overall sentiment score is 0.5, indicating a positive outlook.

### 2. TOP THREE RISKS
Based on both quantitative and qualitative analysis, the top three risks associated with Apple Inc. (AAPL) are:

1. **Investment Generation Risk**: As mentioned in the qualitative context, the main risks lie in how quickly A

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/tmp/ipykernel_10528/2350820784.py:224: ResourceWarning: unclosed file <_io.TextIOWrapper name='agent_trace.jsonl' mode='r' encoding='utf-8'>
  print(f"\n📁 agent_trace.jsonl entries: {sum(1 for _ in open(TRACE_FILE))}")
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and sc

In [10]:
from google.colab import files
files.download('agent_trace.jsonl')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag